# Complete-layer relative h0 calibration

This notebook calibrates the starting tile size `h0` for the complete-layer adaptive cube filler. It sweeps several target cell counts and several geometry ratios, then reports `h0` in relative form so the result can be reused for particles of different absolute size.

For each completed run, the target-cell criterion is:

```python
final_cells >= (1 - TARGET_LOWER_TOLERANCE) * target_cells
```

With the default tolerance this accepts results at or above 90% of the requested target. Among valid rows, the selected candidate is the one with the smallest absolute cell-count error; fill fraction is only a tie-breaker.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SINGLE_GRAIN_DIR = Path.cwd()
if not (SINGLE_GRAIN_DIR / 'utils').is_dir():
    SINGLE_GRAIN_DIR = Path('python/experiments/single_grain').resolve()
sys.path.insert(0, str(SINGLE_GRAIN_DIR))

from utils.geometry import (
    Cylinder,
    Ellipsoid,
    HexagonalPrism,
    Sphere,
    calibrate_h0_sweep,
    estimate_h0_from_volume,
    select_best_h0_records,
)

plt.rcParams.update({'figure.dpi': 115})


## Sweep setup

Targets are `2^8` through `2^12`. Each geometry case defines a characteristic length used to make `h0` relative:

- sphere: `h0 / radius`
- cylinder: `h0 / radius`
- ellipsoid: `h0 / major_semi_axis`
- hexagonal prism: `h0 / side_length`

The absolute dimensions below only set the numerical test domains. Because the output is relative, the selected values can be rescaled as `h0 = h0_relative * characteristic_length`.

In [ ]:
NM = 1e-9

TARGET_VALUES = np.array([2**8, 2**9, 2**10, 2**11, 2**12], dtype=int)
TARGET_LOWER_TOLERANCE = 0.10
EPS = 1e-18
ETA = 0.75
MAX_DEPTH = 8
H_MIN_RATIO = 1 / 16
QUEUE_POLICY = 'symmetric_priority'
GRID_SHIFTS = None  # set to 'half_step' to test 27 run-level shifts per h0
MIN_INSIDE_FRACTION = 1.0
INSIDE_FRACTION_SAMPLES = 3

# Multipliers around the analytic volume estimate. Increase this list for a
# denser calibration around a promising relative h0.
H0_MULTIPLIERS = np.array([0.65, 0.75, 0.85, 0.95, 1.05, 1.20, 1.35, 1.55])

BASE_RADIUS = 40 * NM
BASE_MAJOR_AXIS = 50 * NM
BASE_SIDE_LENGTH = 40 * NM

geometry_cases = []

def add_case(case_key, family, label, ratio_label, domain, scale_length, scale_name):
    geometry_cases.append({
        'case_key': case_key,
        'family': family,
        'label': label,
        'ratio_label': ratio_label,
        'domain': domain,
        'scale_length': float(scale_length),
        'scale_name': scale_name,
    })

add_case(
    'sphere_r',
    'Sphere',
    'Sphere',
    'sphere',
    Sphere(center=[0.0, 0.0, 0.0], radius=BASE_RADIUS),
    BASE_RADIUS,
    'radius',
)

for height_radius_ratio in [1.0, 2.0, 4.0]:
    add_case(
        f'cylinder_hr_{height_radius_ratio:g}',
        'Cylinder',
        f'Cylinder H/R={height_radius_ratio:g}',
        f'H/R={height_radius_ratio:g}',
        Cylinder(
            center=[0.0, 0.0, 0.0],
            radius=BASE_RADIUS,
            length=height_radius_ratio * BASE_RADIUS,
            axis='z',
        ),
        BASE_RADIUS,
        'radius',
    )

for b_over_a, c_over_a in [(0.8, 0.6), (0.6, 0.4), (0.5, 0.25)]:
    add_case(
        f'ellipsoid_ba_{b_over_a:g}_ca_{c_over_a:g}',
        'Ellipsoid',
        f'Ellipsoid b/a={b_over_a:g}, c/a={c_over_a:g}',
        f'b/a={b_over_a:g}, c/a={c_over_a:g}',
        Ellipsoid(
            center=[0.0, 0.0, 0.0],
            semi_axes=(BASE_MAJOR_AXIS, b_over_a * BASE_MAJOR_AXIS, c_over_a * BASE_MAJOR_AXIS),
        ),
        BASE_MAJOR_AXIS,
        'major semi-axis',
    )

for height_side_ratio in [0.5, 1.0, 2.0]:
    add_case(
        f'hex_hs_{height_side_ratio:g}',
        'Hexagonal prism',
        f'Hexagonal prism H/s={height_side_ratio:g}',
        f'H/s={height_side_ratio:g}',
        HexagonalPrism(
            center=[0.0, 0.0, 0.0],
            side_length=BASE_SIDE_LENGTH,
            height=height_side_ratio * BASE_SIDE_LENGTH,
            axis='z',
        ),
        BASE_SIDE_LENGTH,
        'side length',
    )

print(f'{len(geometry_cases)} geometry cases')
for case in geometry_cases:
    h0_estimate = estimate_h0_from_volume(case['domain'], int(TARGET_VALUES[2]), eta=ETA)
    print(
        f"{case['label']:<34} scale={case['scale_name']:<16} "
        f"analytic h0/scale at target={TARGET_VALUES[2]}: {h0_estimate / case['scale_length']:.4f}"
    )


## Run complete-layer calibration

This cell performs `len(geometry_cases) * len(TARGET_VALUES) * len(H0_MULTIPLIERS)` complete-layer fills. With the defaults, that is 400 runs. Enabling `GRID_SHIFTS = 'half_step'` multiplies this by 27, so use that only when you specifically want shift optimization.

In [ ]:
calibration_rows = []
for case in geometry_cases:
    print(f"Calibrating {case['label']} ...")
    rows = calibrate_h0_sweep(
        case['domain'],
        target_values=TARGET_VALUES,
        h0_multipliers=H0_MULTIPLIERS,
        h_min_ratio=H_MIN_RATIO,
        max_depth=MAX_DEPTH,
        eps=EPS,
        queue_policy=QUEUE_POLICY,
        eta=ETA,
        target_lower_tolerance=TARGET_LOWER_TOLERANCE,
        complete_layers=True,
        grid_shifts=GRID_SHIFTS,
        min_inside_fraction=MIN_INSIDE_FRACTION,
        inside_fraction_samples=INSIDE_FRACTION_SAMPLES,
    )
    for row in rows:
        # Override the generic domain shape so each ratio is selected separately.
        row['shape'] = case['case_key']
        row['case_key'] = case['case_key']
        row['family'] = case['family']
        row['display_shape'] = case['label']
        row['ratio_label'] = case['ratio_label']
        row['scale_length'] = case['scale_length']
        row['scale_name'] = case['scale_name']
        row['h0_relative'] = row['h0'] / case['scale_length']
        row['analytic_h0_relative'] = row['analytic_h0'] / case['scale_length']
        row['final_smallest_tile_size_relative'] = (
            None if row['final_smallest_tile_size'] is None
            else row['final_smallest_tile_size'] / case['scale_length']
        )
    calibration_rows.extend(rows)

best_rows = select_best_h0_records(calibration_rows)
# Keep a stable display order in later tables/plots.
case_order = {case['case_key']: index for index, case in enumerate(geometry_cases)}
best_rows = sorted(best_rows, key=lambda row: (case_order[row['case_key']], row['target']))

print(f'Generated {len(calibration_rows)} calibration rows.')
print(f'Selected {len(best_rows)} best rows: one per geometry case and target.')


## Best rows by target

These are the selected candidate `h0` values for each geometry ratio and target cell count. The main reusable result is `h0 / scale`.

In [ ]:
def print_table(records, columns, title=None):
    if title:
        print(title)
    if not records:
        print('(no records)')
        return
    text_rows = []
    for record in records:
        row = []
        for key, header, formatter in columns:
            value = record.get(key)
            row.append(str(formatter(value)) if formatter else str(value))
        text_rows.append(row)
    widths = [len(header) for _key, header, _formatter in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    header = '  '.join(header.ljust(width) for width, (_key, header, _formatter) in zip(widths, columns))
    print(header)
    print('-' * len(header))
    for row in text_rows:
        print('  '.join(value.ljust(width) for value, width in zip(row, widths)))

def fmt_float(value, digits=4):
    return 'n/a' if value is None else f'{value:.{digits}f}'

best_columns = [
    ('family', 'Family', None),
    ('ratio_label', 'Ratio', None),
    ('target', 'Target', lambda value: f'{value:d}'),
    ('h0_relative', 'h0 / scale', lambda value: fmt_float(value, 5)),
    ('h0_multiplier', 'h0 / analytic', lambda value: fmt_float(value, 2)),
    ('final_cells', 'Cells', lambda value: f'{value:d}'),
    ('relative_count_error', 'Rel. error', lambda value: f'{value:+.3f}'),
    ('target_status', 'Status', None),
    ('target_accepted', 'Valid', lambda value: 'yes' if value else 'no'),
    ('fill_fraction', 'Fill', lambda value: fmt_float(value, 3)),
    ('layers_used', 'Layers', lambda value: f'{value:d}'),
]

print_table(best_rows, best_columns, 'Best relative h0 for each target and geometry ratio')


## Aggregate recommendations

For a single reusable number per geometry ratio, this table reports the median selected `h0 / scale` across all tested targets, plus the min/max range over targets. Use the per-target table above when the target count is known and you want the most specific value.

In [ ]:
summary_rows = []
for case in geometry_cases:
    rows = [row for row in best_rows if row['case_key'] == case['case_key']]
    h0_values = np.array([row['h0_relative'] for row in rows], dtype=float)
    abs_errors = np.array([abs(row['relative_count_error']) for row in rows], dtype=float)
    fill_values = np.array([np.nan if row['fill_fraction'] is None else row['fill_fraction'] for row in rows], dtype=float)
    summary_rows.append({
        'family': case['family'],
        'ratio_label': case['ratio_label'],
        'scale_name': case['scale_name'],
        'h0_relative_median': float(np.median(h0_values)),
        'h0_relative_min': float(np.min(h0_values)),
        'h0_relative_max': float(np.max(h0_values)),
        'valid_targets': sum(bool(row['target_accepted']) for row in rows),
        'mean_abs_relative_error': float(np.mean(abs_errors)),
        'mean_fill_fraction': None if np.all(np.isnan(fill_values)) else float(np.nanmean(fill_values)),
    })

summary_columns = [
    ('family', 'Family', None),
    ('ratio_label', 'Ratio', None),
    ('scale_name', 'Scale', None),
    ('h0_relative_median', 'median h0/scale', lambda value: fmt_float(value, 5)),
    ('h0_relative_min', 'min', lambda value: fmt_float(value, 5)),
    ('h0_relative_max', 'max', lambda value: fmt_float(value, 5)),
    ('valid_targets', 'valid targets', lambda value: f'{value:d}/{len(TARGET_VALUES)}'),
    ('mean_abs_relative_error', 'mean |err|', lambda value: fmt_float(value, 3)),
    ('mean_fill_fraction', 'mean fill', lambda value: fmt_float(value, 3)),
]

print_table(summary_rows, summary_columns, 'Aggregate relative h0 recommendations')


## Figures

The first figure shows the selected relative `h0` versus target cell count for each geometry family and ratio. The second figure shows quality diagnostics for the selected rows: relative count error, filled volume fraction, and refinement layers. The third figure shows the candidate landscape, which is useful for spotting whether the multiplier sweep is wide enough.

In [ ]:
families = list(dict.fromkeys(case['family'] for case in geometry_cases))
target_exponents = {int(target): int(np.log2(target)) for target in TARGET_VALUES}

fig, axes = plt.subplots(1, len(families), figsize=(4.2 * len(families), 3.6), sharey=False)
if len(families) == 1:
    axes = [axes]

for ax, family in zip(axes, families):
    family_cases = [case for case in geometry_cases if case['family'] == family]
    for case in family_cases:
        rows = [row for row in best_rows if row['case_key'] == case['case_key']]
        x = [target_exponents[row['target']] for row in rows]
        y = [row['h0_relative'] for row in rows]
        ax.plot(x, y, marker='o', label=case['ratio_label'])
    ax.set_title(family)
    ax.set_xlabel('log2(target cells)')
    ax.set_ylabel('best h0 / scale')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()

fig, axes = plt.subplots(len(families), 3, figsize=(12, 3.0 * len(families)), sharex=True)
if len(families) == 1:
    axes = axes.reshape(1, -1)

for row_index, family in enumerate(families):
    family_cases = [case for case in geometry_cases if case['family'] == family]
    for case in family_cases:
        rows = [row for row in best_rows if row['case_key'] == case['case_key']]
        x = [target_exponents[row['target']] for row in rows]
        axes[row_index, 0].plot(x, [row['relative_count_error'] for row in rows], marker='o', label=case['ratio_label'])
        axes[row_index, 1].plot(x, [row['fill_fraction'] for row in rows], marker='o', label=case['ratio_label'])
        axes[row_index, 2].plot(x, [row['layers_used'] for row in rows], marker='o', label=case['ratio_label'])
    axes[row_index, 0].axhline(0.0, color='black', linestyle='--', linewidth=1)
    axes[row_index, 0].axhline(-TARGET_LOWER_TOLERANCE, color='black', linestyle=':', linewidth=1)
    axes[row_index, 0].set_ylabel(f'{family}\nrelative error')
    axes[row_index, 1].set_ylabel('fill fraction')
    axes[row_index, 2].set_ylabel('layers')
    for ax in axes[row_index]:
        ax.grid(alpha=0.3)
        ax.set_xlabel('log2(target cells)')
    axes[row_index, 2].legend(fontsize=8)
fig.tight_layout()

fig, axes = plt.subplots(len(families), len(TARGET_VALUES), figsize=(3.2 * len(TARGET_VALUES), 3.0 * len(families)), sharex=False, sharey=True)
if len(families) == 1:
    axes = axes.reshape(1, -1)

for family_index, family in enumerate(families):
    for target_index, target in enumerate(TARGET_VALUES):
        ax = axes[family_index, target_index]
        for case in [case for case in geometry_cases if case['family'] == family]:
            rows = sorted(
                [row for row in calibration_rows if row['case_key'] == case['case_key'] and row['target'] == int(target)],
                key=lambda row: row['h0_relative'],
            )
            ax.plot(
                [row['h0_relative'] for row in rows],
                [row['relative_count_error'] for row in rows],
                marker='.',
                linewidth=1,
                label=case['ratio_label'],
            )
        ax.axhline(0.0, color='black', linestyle='--', linewidth=0.8)
        ax.axhline(-TARGET_LOWER_TOLERANCE, color='black', linestyle=':', linewidth=0.8)
        ax.set_title(f'{family}, N={target}')
        ax.set_xlabel('h0 / scale')
        if target_index == 0:
            ax.set_ylabel('relative error')
        ax.grid(alpha=0.3)
    axes[family_index, -1].legend(fontsize=7)
fig.tight_layout()


## Copyable best relative h0 values

The first block prints one best value for each geometry ratio and target. The second block prints one aggregate value per geometry ratio, using the median over tested targets.

In [ ]:
print('Per-target best relative h0')
for row in best_rows:
    print(
        f"{row['family']:<16} {row['ratio_label']:<18} "
        f"target={row['target']:<5d} "
        f"h0_over_{row['scale_name'].replace(' ', '_')}={row['h0_relative']:.6f} "
        f"cells={row['final_cells']:<6d} "
        f"rel_error={row['relative_count_error']:+.3f} "
        f"valid={row['target_accepted']} "
        f"fill={row['fill_fraction']:.3f}"
    )

print('\nAggregate median relative h0')
for row in summary_rows:
    print(
        f"{row['family']:<16} {row['ratio_label']:<18} "
        f"h0_over_{row['scale_name'].replace(' ', '_')}={row['h0_relative_median']:.6f} "
        f"range=[{row['h0_relative_min']:.6f}, {row['h0_relative_max']:.6f}] "
        f"valid_targets={row['valid_targets']}/{len(TARGET_VALUES)} "
        f"mean_abs_error={row['mean_abs_relative_error']:.3f}"
    )
